# QLoRA fine-tune of Qwen2.5-3B-Instruct (run on Google Colab)

This notebook trains a small **LoRA adapter** on a **free Colab T4 GPU**, then downloads it so you can run
it locally in the main `local_llm_lab.ipynb`.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

We use [Unsloth](https://github.com/unslothai/unsloth), which makes QLoRA fast and memory-light enough for
the free tier.

## 1. Install dependencies (Colab)

In [ ]:
# Takes a couple of minutes on first run.
#
# IMPORTANT: if you ran earlier attempts in this session, FIRST do
#   Runtime -> Disconnect and delete runtime
# then run this on the fresh VM, so a stale old trl can't linger.
#
# Why the version pins: Colab ships a very new transformers (5.x). Modern Unsloth caps
# transformers at <=5.5.0, so a plain `pip install unsloth` BACKTRACKS to an ancient
# Unsloth that accepts the newer transformers but pins trl<0.9 — which lacks SFTConfig
# ("cannot import name 'SFTConfig' from 'trl'"). Constraining transformers/trl into
# modern Unsloth's supported window forces a CURRENT Unsloth + a trl that has the
# SFTConfig / processing_class API the training cell uses.
# (Verified locally: transformers 5.5.0 + trl 0.24.0 train with this exact API.)
!pip install -q --upgrade unsloth unsloth_zoo "transformers>=4.51.3,<=5.5.0" "trl>=0.18.2,<=0.24.0" "peft>=0.18.0"

# Import unsloth FIRST so it can patch transformers/trl for its 2x speedup.
import unsloth
import sys, torch, transformers, trl
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | "
      f"transformers {transformers.__version__} | trl {trl.__version__} | unsloth {unsloth.__version__}")
from trl import SFTConfig   # fail fast here if anything is still stale

# NOTE: a pip "ERROR ... cuda-python ... cuda-bindings ... incompatible" line above is
# harmless Colab noise about its own preinstalled CUDA packages — it does not affect training.

## 2. Confirm the GPU

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Runtime to T4 GPU!")

## 3. Load the base model in 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length  = max_seq_length,
    dtype           = None,    # auto
    load_in_4bit    = True,    # QLoRA
)

# Attach LoRA adapters (these are the only weights we train)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Point at the Q&A file in your repo

Instead of manually uploading a file, this pulls your training data **straight from your GitHub repo**, so
the loop is just: edit the file locally → `git push` → run this notebook. Set `DATA_FILE` to whichever file
you want to train on.

The file can be either:
- **JSONL** — one `{"messages": [...]}` record per line (what `finetune/data/train.jsonl` already is), or
- **Plain Q&A text** — blank-line-separated blocks, each with a `Q:` line and an `A:` line.

**Private repo?** Create a fine-grained GitHub token (repo → *Contents: Read-only*) and add it in Colab under
the **🔑 Secrets** panel (left sidebar) as `GH_TOKEN`, with notebook access enabled. If your repo is public,
leave it unset and this still works.

In [ ]:
# ====== ONE-CLICK CONFIG: which Q&A file in your repo to train on ======
REPO      = "andyhudson80/local-llm-lab"
BRANCH    = "main"
DATA_FILE = "finetune/data/train.jsonl"   # <-- change this to train on a different file
# =======================================================================

import urllib.request, json, re

# Optional GitHub token (needed only for PRIVATE repos). Add it in Colab: 🔑 Secrets -> GH_TOKEN
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    GH_TOKEN = None

# Fetch the raw file from GitHub (works for private repos when GH_TOKEN is set).
api = f"https://api.github.com/repos/{REPO}/contents/{DATA_FILE}?ref={BRANCH}"
headers = {"Accept": "application/vnd.github.raw", "User-Agent": "colab-finetune"}
if GH_TOKEN:
    headers["Authorization"] = f"Bearer {GH_TOKEN}"
raw = urllib.request.urlopen(urllib.request.Request(api, headers=headers)).read().decode("utf-8")

# Accept either JSONL ({"messages": [...]} per line) OR a plain-text Q&A file:
#   - examples separated by a line of three or more dashes (---)
#   - each example has a `Q:` line and an `A:` line; Q and A may span multiple lines
#   - full-line # comments before the first Q:/A: are ignored
def to_records(text):
    text = text.strip()
    if text.startswith("{"):                                   # JSONL path
        return [json.loads(l) for l in text.splitlines() if l.strip()]
    recs = []                                                  # plain-text path
    for chunk in re.split(r"(?m)^\s*-{3,}\s*$", text):
        q_lines, a_lines, mode = [], [], None
        for ln in chunk.splitlines():
            s = ln.strip()
            if mode is None and s.startswith("#"):
                continue
            if s[:2].lower() == "q:":   mode, _ = "q", q_lines.append(s[2:].strip())
            elif s[:2].lower() == "a:": mode, _ = "a", a_lines.append(s[2:].strip())
            elif mode == "q":           q_lines.append(ln)
            elif mode == "a":           a_lines.append(ln)
        q, a = "\n".join(q_lines).strip(), "\n".join(a_lines).strip()
        if q and a:
            recs.append({"messages": [{"role": "user", "content": q},
                                      {"role": "assistant", "content": a}]})
    return recs

records = to_records(raw)
fname = "train.jsonl"
with open(fname, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Loaded {len(records)} examples from {REPO}@{BRANCH}/{DATA_FILE}")
print("First example:\n", json.dumps(records[0], ensure_ascii=False, indent=2) if records else "(none)")
assert records, "No examples parsed - check DATA_FILE path and format."

## 5. Build the training set

We apply Qwen's chat template to each `{"messages": [...]}` record to get a single training string.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=fname, split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = ds.map(format_chat)
print("examples:", len(ds))
print(ds[0]["text"][:400])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

# Recent TRL/Transformers API:
#   - `tokenizer=` is now `processing_class=`
#   - `dataset_text_field` and the sequence length live in SFTConfig
#     (and `max_seq_length` was renamed to `max_length`)
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,            # small dataset -> a few dozen steps is plenty
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer.train()

## 7. Quick sanity check

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"What is the capital of France?"}]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True).to("cuda")
out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 8. Save the adapter and download it

This saves just the small LoRA adapter (a few MB), zips it, and downloads it. Unzip it into
`models/adapters/qwen2.5-3b-tldr/` in your local project.

In [ ]:
# Cast the small LoRA weights to fp16 before saving — roughly halves the adapter file
# (~120 MB fp32 -> ~60 MB fp16). It still loads fine on the bf16 base model locally.
import torch
for _n, _p in model.named_parameters():
    if "lora_" in _n and _p.dtype == torch.float32:
        _p.data = _p.data.to(torch.float16)

model.save_pretrained("qwen2.5-3b-tldr")       # adapter only (now ~60 MB)
tokenizer.save_pretrained("qwen2.5-3b-tldr")

import shutil
shutil.make_archive("qwen2.5-3b-tldr", "zip", "qwen2.5-3b-tldr")

from google.colab import files
files.download("qwen2.5-3b-tldr.zip")

## Optional: export to GGUF for Ollama

If you'd rather run the fine-tuned model through Ollama (fast path), Unsloth can merge + export to GGUF:

```python
model.save_pretrained_gguf("qwen2.5-3b-tldr-gguf", tokenizer, quantization_method="q4_k_m")
```

Then create an Ollama `Modelfile` pointing at the `.gguf` and `ollama create`. See Unsloth's docs.